### 1. Data Loading and Data Engineering



In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_squared_error


# Load the data
df = pd.read_csv('../survey_results_public.csv')



C:\Users\ReDI\AppData\Local\Temp\ipykernel_13916\4017044811.py:16: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../survey_results_public.csv')


In [60]:
#Select only developers by profession and also in DACH region
df = df[(df['ConvertedCompYearly'] > 0) &
        (df['Country'].isin(['Germany', 'Austria', 'Switzerland'])) &
        (df['MainBranch'] == 'I am a developer by profession')
         ]


print("Data shape:", df.shape)
df.head(5)

Data shape: (2343, 172)


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
35,36,I am a developer by profession,25-34 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,10.0,"Yes, I am not new to coding but am learning ne...",AI CodeGen tools or AI-enabled apps,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,problem solving,95132.0,6.0
49,50,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,Volunteering (regularly),6.0,"Yes, I am not new to coding but am learning ne...",Books / Physical media;Videos (not associated ...,"Yes, I learned how to use AI-enabled tools for...",...,Ollama;LangChain,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,Understanding context,117793.0,8.0
50,51,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,None of the above,10.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I’m stuck...,"Soft skills, and the capacity to have complete...",75410.0,6.0
55,56,I am a developer by profession,18-24 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,NaN,3.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","No, I learned something that was not related t...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,NaN,46406.0,9.0
99,100,I am a developer by profession,45-54 years old,"Secondary school (e.g. American high school, G...",Employed,None of the above,26.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Ollama,NaN,Grafana + Prometheus,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,Experience is the key when designing and devel...,145018.0,10.0


***
### 2. Data Splitting

Split the processed data into training and testing sets.

In [61]:
# Define the target variable (y) and features (X)
X = df.drop('ConvertedCompYearly', axis=1)
y = df['ConvertedCompYearly']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

Shape of X_train: (1874, 171)
Shape of X_test: (469, 171)


***
### 3. Data Preprocessing Pipeline

Fill in missing values using ColumnTransformer

In [62]:
# Features to impute with the median (numerical)
numerical_features = ['JobSat']

# Features to impute with the mode and One-Hot Encode (categorical/ordinal)
categorical_features = ['Age','EdLevel', 'DevType', 'RemoteWork','Country']

# A. Numerical Pipeline: Impute with Median
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

# B. Categorical Pipeline: Impute with Mode, then One-Hot Encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')) 
])
 
# C. Combine all steps using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop' # Drop all other columns
)


pipeline = Pipeline ([
    ('preprocessor',preprocessor),
    ('model', Lasso(alpha=0.1, max_iter=100000))
])


kf = KFold(n_splits=10, shuffle=True, random_state=42)

# This gives predictions as if each point was "test" in its fold
cv_predictions = cross_val_predict(
    pipeline,
    X_train,
    y_train,
    cv=kf
)

# Metrics on CV predictions
print("Cross-Validation Metrics (on training data)")

mae = mean_absolute_error(y_train, cv_predictions)
mse = mean_squared_error(y_train, cv_predictions)
rmse = root_mean_squared_error(y_train, cv_predictions)
r2 = r2_score(y_train, cv_predictions)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)






Cross-Validation Metrics (on training data)
MAE: 31938.340495650307
MSE: 4092446369.075308
RMSE: 63972.23123414806
R2: 0.1216566808025008


***
### 4. Plotting Functions

Functions to help visualize the model's performance and structure.

***
### 5. Model Training and Evaluation


In [63]:
# Fit actual model

pipeline.fit(X_train, y_train)

# Evaluation
prediction = pipeline.predict(X_test)

print("\n=== Test Set Metrics ===")

mae = mean_absolute_error(y_test, prediction)
mse = mean_squared_error(y_test, prediction)
rmse = root_mean_squared_error(y_test, prediction)
r2 = r2_score(y_test, prediction)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)



=== Test Set Metrics ===
MAE: 26473.412579215055
MSE: 1913139720.639626
RMSE: 43739.45267878447
R2: 0.3242510614138818


In [64]:
# Train the model using the model defined in the pipeline (LinearRegression)
pipeline.fit(X_train, y_train)

# Predict and Evaluate
prediction = pipeline.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)


Mean Absolute Error:  26473.412579215055
Mean Squared Error:  1913139720.639626
Root Mean Squared Error:  43739.45267878447
r2 score: 0.3242510614138818


In [65]:
# Train the model
model = DecisionTreeRegressor(max_depth=1)
model.fit(X_train, y_train)

# Predict and Evaluate
prediction = model.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)

ValueError: could not convert string to float: 'I am a developer by profession'

In [ ]:
# Train the model
model = Lasso(alpha=0.1, max_iter=100000)
model.fit(X_train, y_train)

# Predict and Evaluate
prediction = model.predict(X_test)

#Evaluate Model Accuracy using root_mean_squared_error, r2_score, mean_absolute_error
mae = mean_absolute_error(y_test,prediction)
mse = mean_squared_error(y_test,prediction)
rmse = root_mean_squared_error(y_test,prediction)
r2 = r2_score(y_test,prediction) 

print("Mean Absolute Error: ",mae)
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", rmse)
print("r2 score:", r2)

Mean Absolute Error:  26473.412579215055
Mean Squared Error:  1913139720.639626
Root Mean Squared Error:  43739.45267878447
r2 score: 0.3242510614138818
